# 02. Baseline — LightGBM / XGBoost / CatBoost

特徴量エンジニアリングなしで、数値7列 + カテゴリ6列をそのまま投入する。
**カテゴリはラベルエンコードせず、各ライブラリのネイティブなカテゴリ対応に渡す。**

CV は全モデル共通で `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`。
この分割を全モデルで揃えることで、あとから OOF 同士をアンサンブルできる。

In [ ]:
import os, sys
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

TARGET = "Will_Buy_EV"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

NUM_COLS = [c for c in train.select_dtypes(include=[np.number]).columns if c != "id"]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + ["id", TARGET]]
FEATURES = NUM_COLS + CAT_COLS

X, X_test = train[FEATURES].copy(), test[FEATURES].copy()
y = (train[TARGET] == "Yes").astype(int)

# train/test で共通のカテゴリ集合を定義しておく(未知カテゴリの不整合を防ぐ)
for c in CAT_COLS:
    cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
    X[c] = pd.Categorical(train[c], categories=cats)
    X_test[c] = pd.Categorical(test[c], categories=cats)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(len(FEATURES), "features")

## 共通の CV ループ

fold ごとに学習して OOF 予測を作り、test 予測は fold 平均にする。

In [ ]:
def run_cv(make_model, X, X_test, y, fit_kwargs=None, n_splits=5):
    oof = np.zeros(len(X))
    pred = np.zeros(len(X_test))
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = make_model()
        model.fit(X.iloc[tr], y.iloc[tr], **(fit_kwargs or {}))
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        pred += model.predict_proba(X_test)[:, 1] / n_splits
        print(f"  fold {fold} AUC: {roc_auc_score(y.iloc[va], oof[va]):.5f}")
    print(f"OOF AUC: {roc_auc_score(y, oof):.5f}")
    return oof, pred

## LightGBM — `category` dtype をそのまま渡す

所要 1〜2 分。

In [ ]:
from lightgbm import LGBMClassifier

oof_lgbm, pred_lgbm = run_cv(
    lambda: LGBMClassifier(random_state=42, verbosity=-1), X, X_test, y
)

## XGBoost — `enable_categorical=True`

所要 2〜3 分。

In [ ]:
from xgboost import XGBClassifier

oof_xgb, pred_xgb = run_cv(
    lambda: XGBClassifier(random_state=42, tree_method="hist", enable_categorical=True),
    X, X_test, y
)

## CatBoost — `cat_features` に列名を渡す

**所要 20 分以上**。時間がないときはこのセルを飛ばしてよい。

In [ ]:
from catboost import CatBoostClassifier

Xc, Xc_test = X.copy(), X_test.copy()
for c in CAT_COLS:          # CatBoost は文字列のまま渡す
    Xc[c] = Xc[c].astype(str)
    Xc_test[c] = Xc_test[c].astype(str)

oof_cat, pred_cat = run_cv(
    lambda: CatBoostClassifier(random_state=42, verbose=False, allow_writing_files=False),
    Xc, Xc_test, y, fit_kwargs={"cat_features": CAT_COLS}
)

## ベースラインの結果

| モデル | OOF AUC | Public LB |
|---|---|---|
| LightGBM | 0.94123 | 0.94093 |
| XGBoost | 0.94124 | 0.94152 |
| CatBoost | 0.94156 | 0.94170 |

3モデルともほぼ同水準。ここから特徴量エンジニアリングで +0.003〜0.004 を積む(→ `04_train_and_evaluate.ipynb`)。